In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/customer_events.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (57094, 12)

Columns:
['event_id', 'user_id', 'session_id', 'event_timestamp', 'event_type', 'product_id', 'category', 'device', 'country', 'traffic_source', 'experiment_variant', 'price']

First 5 rows:


,event_id,user_id,session_id,event_timestamp,event_type,product_id,category,device,country,traffic_source,experiment_variant,price
0,1,15827,sess_15827_1,2025-01-01 00:14:35,visit,NaN,NaN,Mobile,USA,Direct,Control,NaN
1,2,2067,sess_2067_3,2025-01-01 00:16:45,visit,NaN,NaN,Desktop,USA,Social,Variant,NaN
2,3,15827,sess_15827_1,2025-01-01 00:16:58,product_view,SPORT002,Sports,Mobile,USA,Direct,Control,159.66
3,4,15827,sess_15827_1,2025-01-01 00:17:54,add_to_cart,SPORT002,Sports,Mobile,USA,Direct,Control,159.66
4,5,9337,sess_9337_1,2025-01-01 00:22:44,visit,NaN,NaN,Mobile,UK,Social,Control,NaN


In [ ]:
# checking for missing values, if there are any.
missingValues = df.isnull().sum()
print("Missing values by column:")
print(missingValues)

Missing values by column:
event_id                  0
user_id                   0
session_id                0
event_timestamp           0
event_type                0
product_id            20000
category              20000
device                    0
country                   0
traffic_source            0
experiment_variant        0
price                 20000
dtype: int64


In [4]:
duplicate_events = df.duplicated().sum()
print("Duplicate rows:", duplicate_events)

duplicate_event_ids = df["event_id"].duplicated().sum()
print("Duplicate event IDs:", duplicate_event_ids)

Duplicate rows: 0
Duplicate event IDs: 0


In [5]:
event_order = {
    "visit": 1,
    "product_view": 2,
    "add_to_cart": 3,
    "checkout": 4,
    "purchase": 5,
}

df["event_order"] = df["event_type"].map(event_order)

session_check = (
    df.sort_values(["session_id", "event_timestamp"])
      .groupby("session_id")["event_order"]
      .apply(lambda x: (x.diff().dropna() < 0).sum())
)

print("Sessions with events moving backwards:", (session_check > 0).sum())

Sessions with events moving backwards: 0


In [6]:
print("Event types:")
print(sorted(df["event_type"].unique()))

print("\nDevices:")
print(sorted(df["device"].unique()))

print("\nCountries:")
print(sorted(df["country"].unique()))

print("\nTraffic sources:")
print(sorted(df["traffic_source"].unique()))

print("\nExperiment variants:")
print(sorted(df["experiment_variant"].unique()))

print("\nCategories:")
print(sorted(df["category"].dropna().unique()))

Event types:
['add_to_cart', 'checkout', 'product_view', 'purchase', 'visit']

Devices:
['Desktop', 'Mobile', 'Tablet']

Countries:
['Australia', 'Canada', 'Germany', 'UK', 'USA']

Traffic sources:
['Direct', 'Email', 'Organic', 'Paid Search', 'Social']

Experiment variants:
['Control', 'Variant']

Categories:
['Beauty', 'Clothing', 'Electronics', 'Home', 'Sports']


In [7]:
print("Price summary:")
print(df["price"].describe())

Price summary:
count    37094.000000
mean       262.694026
std        252.614205
min         10.080000
25%         98.035000
50%        175.090000
75%        333.770000
max       1199.960000
Name: price, dtype: float64


In [8]:
print("\nPrices below or equal to zero:")
print((df["price"] <= 0).sum())

print("\nMissing prices by event type:")
print(df.groupby("event_type")["price"].apply(lambda x: x.isna().sum()))


Prices below or equal to zero:
0

Missing prices by event type:
event_type
add_to_cart         0
checkout            0
product_view        0
purchase            0
visit           20000
Name: price, dtype: int64


In [9]:
print("Earliest event:", df["event_timestamp"].min())
print("Latest event:", df["event_timestamp"].max())

print("\nEvents before start date:",
      (df["event_timestamp"] < "2025-01-01").sum())

print("Events after end date:",
      (df["event_timestamp"] >= "2025-07-01").sum())

Earliest event: 2025-01-01 00:14:35
Latest event: 2025-06-30 00:04:17

Events before start date: 0
Events after end date: 0


In [10]:
# Check that every session has a visit
sessions_without_visit = (
    df.groupby("session_id")["event_type"]
      .apply(lambda x: "visit" not in x.values)
)

print("Sessions without a visit:", sessions_without_visit.sum())


# Check that every purchase has a checkout in the same session
session_events = (
    df.groupby("session_id")["event_type"]
      .apply(set)
)

purchases_without_checkout = sum(
    "purchase" in events and "checkout" not in events
    for events in session_events
)

print("Purchases without checkout:", purchases_without_checkout)

Sessions without a visit: 0
Purchases without checkout: 0
